In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [ ]:
## data input
data = ""

In [ ]:
bike_df = pd.read_csv('./data/') #TODO:YOUR_CODE_HERE
bike_df.head(5)

In [1]:
%pip install requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import requests
from dotenv import load_dotenv

# .env 파일 불러오기
load_dotenv(r"Backend\key\.env")

BASE_URL = os.getenv("META_BASE_URL")
API_VERSION = os.getenv("META_API_VERSION")
ACCESS_TOKEN = os.getenv("META_ACCESS_TOKEN")


def get_accounts():
    """내 계정(페이지) 목록 가져오기 - Postman에서 했던 /me/accounts 요청"""
    url = f"{BASE_URL}/{API_VERSION}/me/accounts"
    params = {
        "fields": "id,name,access_token",
        "access_token": ACCESS_TOKEN
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:
        data = response.json()
        return data.get("data", [])
    else:
        print(f"에러 발생: {response.status_code}")
        print(response.text)
        return None


if __name__ == "__main__":
    accounts = get_accounts()

    if accounts:
        for account in accounts:
            print(f"페이지 이름: {account.get('name')}")
            print(f"페이지 ID: {account.get('id')}")
            print(f"페이지 토큰: {account.get('access_token')[:20]}...")  # 토큰은 일부만 출력
            print("-" * 40)

In [ ]:
# 포스트맨 json 값 카운트 확인하기 (보기 쉽게)
# responseBody가 API 응답(JSON)이라고 가정
const json = pm.response.json();

const hashtagCount = {};

json.data.forEach(item => {
    if (!item.caption) return;

    // #으로 시작하는 해시태그 추출
    const hashtags = item.caption.match(/#[^\s#]+/g);

    if (!hashtags) return;

    hashtags.forEach(tag => {
        // 소문자로 통일하고 싶다면 .toLowerCase() 추가
        tag = tag.trim();

        hashtagCount[tag] = (hashtagCount[tag] || 0) + 1;
    });
});

// 많이 나온 순으로 정렬
const sorted = Object.entries(hashtagCount)
    .sort((a, b) => b[1] - a[1]);

console.log(sorted);

// 보기 좋게 출력
sorted.forEach(([tag, count]) => {
    console.log(`${tag}: ${count}`);
});

In [ ]:
# 카운트 확인하기 (json 방식으로)
# responseBody가 API 응답(JSON)이라고 가정

const json = pm.response.json();

const hashtagCount = {};

json.data.forEach(item => {
    if (!item.caption) return;

    const hashtags = item.caption.match(/#[^\s#]+/g);
    if (!hashtags) return;

    hashtags.forEach(tag => {
        hashtagCount[tag] = (hashtagCount[tag] || 0) + 1;
    });
});

const result = Object.entries(hashtagCount)
    .sort((a, b) => b[1] - a[1])
    .map(([hashtag, count]) => ({
        hashtag,
        count
    }));

console.log(JSON.stringify(result, null, 2));

In [ ]:
# 인스타 api 가져와서 날짜별 저장 코드
import requests
import json
import os
import re
from dotenv import load_dotenv
from datetime import datetime, timedelta


load_dotenv(r"Backend\key\.env")

META_URL = os.getenv("META_BASE_URL") # https://graph.facebook.com
API_VERSION = os.getenv("META_API_VERSION") # v25.0
ACCESS_TOKEN = os.getenv("META_ACCESS_TOKEN")
USER_ID = os.getenv("META_USER_ID")

keyword="황치즈"

class InstagramCollector:

    def __init__(self, ACCESS_TOKEN, USER_ID, API_VERSION, META_URL):

        self.access_token = ACCESS_TOKEN
        self.ig_user_id = USER_ID
        self.api_version = API_VERSION
        self.mt_url = META_URL

    ########################################################
    # hashtag id 가져오기
    ########################################################

    def get_hashtag_id(self, keyword):
        os.makedirs("cache", exist_ok=True)

        cache_file = "cache/hashtag_cache.json"

        if os.path.exists(cache_file):
            with open(cache_file, "r", encoding="utf-8") as f:
                cache = json.load(f)
        else:
            cache = {}

        # 이미 검색한 키워드면 API 호출 안 함
        if keyword in cache:
            return cache[keyword]

        url = f"{self.mt_url}/{self.api_version}/ig_hashtag_search"

        params = {
            "user_id": self.ig_user_id,
            "q": keyword,
            "access_token": self.access_token
        }

        response = requests.get(url, params=params)
        response.raise_for_status()

        result = response.json()

        if len(result["data"]) == 0:
            return None

        hashtag_id = result["data"][0]["id"]

        cache[keyword] = hashtag_id

        with open(cache_file, "w", encoding="utf-8") as f:
            json.dump(cache, f, ensure_ascii=False, indent=4)

        return hashtag_id

    ########################################################
    # 게시글 조회
    ########################################################

    def get_recent_media(self, hashtag_id):

        url = f"{self.mt_url}/{self.api_version}/{hashtag_id}/recent_media"

        params = {
            "user_id": self.ig_user_id,
            "fields": "id,caption,media_type,media_url,timestamp",
            "access_token": self.access_token
        }

        response = requests.get(url, params=params)
        response.raise_for_status()

        return response.json()

    ########################################################
    # 해시태그 카운트
    ########################################################

    def hashtag_counter(self, media):

        counter = {}

        for post in media.get("data", []):

            caption = post.get("caption", "")

            hashtags = re.findall(r"#\S+", caption)

            for tag in hashtags:
                counter[tag] = counter.get(tag, 0) + 1

        counter = dict(
            sorted(
                counter.items(),
                key=lambda x: x[1],
                reverse=True
            )
        )

        return counter
    
    ########################################################
    # 최근 수집 시간 확인
    ########################################################

    def can_collect(self, keyword, minutes=30):

        os.makedirs("cache/media_cache", exist_ok=True)

        file = f"cache/media_cache/{keyword}.json"

        if not os.path.exists(file):
            return True

        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        last = datetime.fromisoformat(data["last_collected"])

        if datetime.now() - last < timedelta(minutes=minutes):
            return False

        return True
    
    ########################################################
    # 마지막 수집 시간 저장
    ########################################################

    def update_collect_time(self, keyword):

        os.makedirs("cache/media_cache", exist_ok=True)

        file = f"cache/media_cache/{keyword}.json"

        with open(file, "w", encoding="utf-8") as f:

            json.dump(
                {
                    "last_collected": datetime.now().isoformat()
                },
                f,
                ensure_ascii=False,
                indent=4
            )

    ########################################################
    # 저장
    ########################################################

    def save_result(self, keyword, hashtag_id, media):

        today = datetime.now().strftime("%Y-%m-%d")

        os.makedirs("data", exist_ok=True)

        filename = f"data/{today}.json"

        hashtag_count = self.hashtag_counter(media)

        keyword_data = {
            "keyword": keyword,
            "collected_at": datetime.now().isoformat(),
            "hashtag_id": hashtag_id,
            "post_count": len(media.get("data", [])),
            "hashtag_total": sum(hashtag_count.values()),
            "hashtags": hashtag_count,
            "posts": media.get("data", [])
        }

        ####################################################
        # 파일 없으면 생성
        ####################################################

        if not os.path.exists(filename):

            result = {
                "date": today,
                "keyword_count": 1,
                "keywords": [
                    {
                        "keyword_no": 1,
                        **keyword_data
                    }
                ]
            }

        ####################################################
        # 있으면 이어쓰기
        ####################################################

        else:

            with open(filename, "r", encoding="utf-8") as f:
                result = json.load(f)

            keyword_no = len(result["keywords"]) + 1

            result["keywords"].append({
                "keyword_no": keyword_no,
                **keyword_data
            })

            result["keyword_count"] = len(result["keywords"])

        with open(filename, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=4)

        print(keyword, "저장 완료")

    ########################################################
    # 실행
    ########################################################

    def collect(self, keyword):

        try:
            minutes = int(os.getenv("COLLECT_INTERVAL", "60"))

            if not self.can_collect(keyword, minutes):
                print(f"{keyword}는 최근 {minutes}분 안에 수집되었습니다.")
                return

            hashtag_id = self.get_hashtag_id(keyword)

            if hashtag_id is None:
                print(f"{keyword} 검색 결과가 없습니다.")
                return

            media = self.get_recent_media(hashtag_id)

            self.save_result(keyword, hashtag_id, media)

            self.update_collect_time(keyword)

        except Exception as e:

            print(f"{keyword} 수집 실패")

            print(e)

if __name__ == "__main__":

    collector = InstagramCollector(
        ACCESS_TOKEN,
        USER_ID,
        API_VERSION,
        META_URL
    )

    collector.collect(keyword)